# Model 2: Supervised Fine-Tuning (SFT)

## Goal
Fine-tune Qwen2.5-1.5B-Instruct on Austrian tax law Q&A pairs, then run inference on all 643 test questions and produce a submission CSV.

## Approach
- load the base instruction model: Qwen2.5-1.5B-Instruct (4-bit quantised via Unsloth)
- add LoRA adapters and fine-tune only those 
- use completion-only loss so the model only learns from the answer tokens, not the question
- hold out 10% of training examples as a validation set
- save the trained adapter to Hugging Face Hub
- run batched inference on all 643 test questions

## Requirements
- Run on Kaggle with a T4 GPU 
- Create a free account at huggingface.co and get a write token from huggingface.co/settings/tokens

In [ ]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

- When using Kaggle T4 x2, we force Python to only see one GPU.
- Unsloth is optimised for single-GPU use and may crash otherwise.

## 1. Install Libraries

In [ ]:
!pip install unsloth --quiet
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps trl peft accelerate bitsandbytes --quiet
!pip install pandas tqdm huggingface_hub --quiet


## 2. Import Libraries

In [ ]:
import os
import math
import time
import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import login
from tqdm import tqdm
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TrainingArguments
from trl import SFTTrainer

## 3. Configuration

Here all static parameters are defined: model, file paths, LoRA settings, training settings, and inference settings.

In [ ]:
# HF credentials
# The  adapter is pushed to HF so it is not lost when the session ends.
# Get your write token from: https://huggingface.co/settings/tokens
HF_TOKEN    = "hf_YOUR_TOKEN_HERE"            
HF_USERNAME = "your-hf-username"              
HF_REPO     = "qwen2.5-1.5b-austrian-tax-sft"

SFT_CSV  = "/kaggle/input/datasets/lucaboesso/trainingdata-sft/SFT.csv"  
OUT_CSV  = "/kaggle/working/sft_results_qwen.csv"

!wget -q https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv
TEST_CSV = "dataset_clean.csv"


# pre-quantised 4-bit version
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

# Training hyperparameters
MAX_SEQ_LEN       = 512   # Maximum token length per training example
NUM_EPOCHS        = 5     # How many times the model sees the full training set
BATCH_SIZE        = 4     # Examples processed per GPU step
GRAD_ACCUM        = 4     # Effective batch size = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE     = 2e-4  # How fast the model updates its weights
LORA_RANK         = 16    # Size of LoRA adapter matrices (higher = more capacity)
VALIDATION_SHARE  = 0.1   # 10% of training examples held out for validation
RANDOM_SEED       = 420

# inference
MAX_NEW_TOKENS = 300  # Maximum tokens the model can generate per answer
BATCH_SIZE_INF = 4    

# --- System prompt ---
# This tells the model what role to play. Used during both training and inference.
SYSTEM_PROMPT = (
    "You are a careful Austrian tax law assistant. "
    "Answer briefly, factually, and clearly in one plain-text paragraph. "
    "Keep your answer under 200 words. "
    "Complete your sentence before stopping. "
    "Do not invent legal references. "
    "Respond in German."
)

TEST_PROMPT = "Was ist die Körperschaftsteuer?"

print("Base model :", BASE_MODEL)
print("Training   :", NUM_EPOCHS, "epochs on", SFT_CSV)
print("Output     :", OUT_CSV)

## 4. GPU Configuration

Training should only start if CUDA is available.
TF32 enabled for faster matrix operations on NVIDIA GPUs.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook needs a GPU.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Torch version:", torch.__version__)

## 5. Hugging Face Login

In [ ]:
login(token=HF_TOKEN)
print("Logged in to Hugging Face.")

## 6. Load and Prepare Training Data

SFT.csv contains 50 Austrian tax Q&A pairs with columns `instruction` and `response`.
Each row is converted into a 3-turn chat: system → user → assistant.
This matches the format Qwen2.5-Instruct was originally trained on.
SFT.csv was created based on Fallsammlung Staringer/Wallig SS26 from the "Institut für Österreichisches und Internationales Steuerrecht" at WU.

In [ ]:
raw = pd.read_csv(
    SFT_CSV,
    sep=';',
    encoding='utf-8-sig',
    skiprows=2,
    header=None,
    names=['instruction', 'response']
)

sft_df = raw.dropna(subset=['instruction', 'response']).reset_index(drop=True)
sft_df['instruction'] = sft_df['instruction'].str.strip()
sft_df['response']    = sft_df['response'].str.strip()

print("Usable rows:", len(sft_df))
print()
print("First example:")
print("  Q:", sft_df['instruction'].iloc[0][:69], "...")
print("  A:", sft_df['response'].iloc[0][:69], "...")

In [ ]:
# Convert each row into a chat-formatted example
examples = []
for _, row in sft_df.iterrows():
    examples.append({
        "instruction": row["instruction"],
        "response":    row["response"],
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": row["instruction"]},
            {"role": "assistant", "content": row["response"]}
        ]
    })

dataset = Dataset.from_list(examples)
print("Dataset size:", len(dataset))

## 7. Train / Validation Split

10% of the training examples are held out as a validation set.
This lets us monitor whether the model is overfitting during training.

In [ ]:
validation_size = max(1, int(len(dataset) * VALIDATION_SHARE)) # At least 1 example is kept for validation

split = dataset.train_test_split(test_size=validation_size, seed=RANDOM_SEED, shuffle=True)
train_dataset = split["train"]
eval_dataset  = split["test"]
print("Train examples     :", len(train_dataset))
print("Validation examples:", len(eval_dataset))

## 8. Load the Base Model

We load Qwen2.5-1.5B in 4-bit quantised format using Unsloth.
4-bit quantisation compresses the model from ~3GB to ~2,4GB VRAM, making it feasible on a T4 GPU with limited compute time.

Then we add LoRA adapters: small trainable matrices inserted into the attention layers.
Only the adapter weights (~18M out of 1.5B parameters) are updated during training.

In [ ]:
print("Loading", BASE_MODEL, "...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,   # Auto-detect: float16 on T4
    load_in_4bit   = True,   # 4-bit quantisation to save VRAM
    token          = HF_TOKEN,
)

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Base model loaded.")
print("VRAM used:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")
print()

In [ ]:
# LoRA adapters addedto the model.
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_RANK * 2,
    lora_dropout   = 0.05,
    bias           = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",  
    random_state   = RANDOM_SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print("Trainable parameters:", f"{trainable/1e6:.1f}M / {total/1e6:.0f}M total ({100*trainable/total:.1f}%)")


## 9. Training Settings

The formatting function converts each batch of examples into strings using the model's chat template.

The key improvement over a standard setup is `completion_only_loss=True`.
This means the loss is only computed on the **assistant's answer tokens**, not on the system prompt or question.
The model therefore only learns from what it is supposed to generate, which is more efficient.

In [ ]:
from trl import SFTConfig

def format_example(example): # converts each batch into a list of strings; because SFTTrainer requires a list 
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )}

train_dataset = train_dataset.map(format_example)
eval_dataset  = eval_dataset.map(format_example)

training_args = SFTConfig(
    output_dir                  = "./checkpoints",
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 5,
    weight_decay                = 0.01,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 5,
    eval_strategy               = "epoch", # Evaluate on validation set after each epoch
    save_strategy               = "epoch",
    save_total_limit            = 2,
    optim                       = "adamw_8bit",
    seed                        = RANDOM_SEED,
    report_to                   = "none",
    max_length                  = MAX_SEQ_LEN,
    packing                     = False,
    completion_only_loss        = True, # compute loss only on assistant answer tokens
    eos_token                   = "<|im_end|>", # Qwen's end-of-turn token
    dataset_text_field          = "text",  # tells the trainer which column to use
)

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    args         = training_args,
    
)

print("Trainer created.")
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM)

## 10. Start Training

The model loops through the training examples for NUM_EPOCHS epochs.
The training loss should decrease from ~3.0 to ~1.3 over the run.
The validation loss is printed after each epoch.

In [ ]:
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

print("Training finished.")
print("Time       :", round(elapsed / 60, 1), "minutes")
print("Final loss :", round(trainer_stats.training_loss, 4))
print("VRAM used  :", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB")

## 11. Save the Adapter to Hugging Face Hub

Only the LoRA adapter is uploaded. Ensures the result is not lost when the Kaggle session ends.

In [ ]:
REPO_ID = f"{HF_USERNAME}/{HF_REPO}"

print("Saving adapter to:", f"https://huggingface.co/{REPO_ID}")
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
print("Adapter saved.")
print()

## 12. Quick Test

A short generation test is run after training.
This checks whether the fine-tuned model produces a sensible answer for a sample tax-law question.

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": TEST_PROMPT},
]

inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = model.generate(
        input_ids    = inputs,
        max_new_tokens = 120,
        do_sample    = False,
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens (exclude the input prompt)
generated = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("Test prompt:")
print(TEST_PROMPT)
print()
print("Model answer:")
print(generated)

## 13. Load Test Dataset

The 643 test questions from the course dataset are loaded.
These are the questions the model needs to answer for the submission.

In [ ]:
input_df = pd.read_csv(TEST_CSV, encoding="utf-8-sig", quotechar='"', on_bad_lines='warn')
input_df.columns = [c.strip().lower() for c in input_df.columns]
input_df['prompt'] = input_df['prompt'].astype(str).str.strip()
input_df['id']     = input_df['id'].astype(str).str.strip()

print("Test questions loaded:", len(input_df))
print()
print(input_df[['id', 'prompt']].head(3).to_string())

## 14. Batch Inference

Instead of generating one answer at a time, questions are processed in batches of 4.
This is faster because of the GPU.

For batched generation, the tokenizer uses left-padding so that all sequences in a batch
end at the same position, and generation starts from there.

In [ ]:
# Build one chat-formatted input string from a single user prompt
def build_chat_text(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
# skip the questions that were already answered
if os.path.exists(OUT_CSV):
    existing_df = pd.read_csv(OUT_CSV, encoding="utf-8-sig")
    done_ids    = set(existing_df["id"].astype(str))
    print("Resume:", len(done_ids), "questions already done.")
else:
    existing_df = pd.DataFrame(columns=["id", "answer"])
    done_ids    = set()

work_df = input_df[~input_df["id"].astype(str).isin(done_ids)].reset_index(drop=True)
print("Remaining:", len(work_df), "/", len(input_df))
print()

if len(work_df) == 0:
    print("Output file already complete.")
    raise SystemExit

# For batched generation, left-padding is required so all sequences in a batch
# end at the same position and new tokens are generated from there
tokenizer.padding_side = "left"

new_results = []
num_batches = math.ceil(len(work_df) / BATCH_SIZE_INF)
start_time  = time.time()

for batch_idx in range(num_batches):
    batch_start = batch_idx * BATCH_SIZE_INF
    batch_end   = min(batch_start + BATCH_SIZE_INF, len(work_df))
    batch_df    = work_df.iloc[batch_start:batch_end]

    ids     = batch_df["id"].tolist()
    prompts = batch_df["prompt"].tolist()

    # Convert each prompt to a chat-formatted string
    chat_texts = [build_chat_text(p) for p in prompts]

    # Tokenize the whole batch together with padding
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = MAX_NEW_TOKENS,
            do_sample         = False,
            use_cache         = True,
            repetition_penalty = 1.05,
            pad_token_id      = tokenizer.pad_token_id,
            eos_token_id      = tokenizer.eos_token_id,
        )

    for i in range(len(ids)): # With left-padding, generated tokens start after the full padded input length
        padded_input_length = inputs["input_ids"].shape[1]
        generated_tokens = outputs[i][padded_input_length:]
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        answer = " ".join(answer.split())  # Flatten to single line
        new_results.append({"id": ids[i], "answer": answer})

    # Save progress after every batch
    temp_df = pd.concat(
        [existing_df, pd.DataFrame(new_results, columns=["id", "answer"])],
        ignore_index=True
    )
    temp_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    processed = batch_end
    elapsed   = time.time() - start_time
    per_item  = elapsed / processed
    eta_min   = (per_item * (len(work_df) - processed)) / 60
    print(f"Batch {batch_idx + 1}/{num_batches} finished | {processed}/{len(work_df)} prompts | ETA approx. {eta_min:.1f} min")

print()
print("Run finished!")
print("Results saved to:", OUT_CSV)

## 15. Verify Output

In [ ]:
final_df = pd.read_csv(OUT_CSV, encoding="utf-8-sig")

print("Total rows :", len(final_df), "(expected", len(input_df), ")")
print("Columns    :", list(final_df.columns), "(expected ['id', 'answer'])")
print("Complete   :", "YES" if len(final_df) == len(input_df) else "NO - missing rows!")
print()
print("First 3 results:")
print(final_df.head(3).to_string())